In [1]:
import sys; sys.path.append("..")
%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd
from src.db_sql import load_clean_data
from src.features import get_xy, split_data, build_preprocessor
from src.models import tune_and_fit
from src.logger_setup import flush_mongo_logs

df = load_clean_data()          # Tabla 1, sin repetir la limpieza
X, y = get_xy(df)
X_train, X_test, y_train, y_test = split_data(X, y)

# comprobación del preprocesamiento
prep = build_preprocessor().fit(X_train)
Xt = prep.transform(X_train.head(1000))
print(Xt.shape, "| NaN restantes:", int(np.isnan(Xt.toarray() if hasattr(Xt, "toarray") else Xt).sum()))
print(prep.get_feature_names_out()[:15])

2026-11-09 22:54:24,757 | INFO     | src.db_sql | Tabla 'datos_limpios' leída: 174116 filas, 20 columnas
2026-11-09 22:54:24,854 | INFO     | src.features | X: (174116, 12) | y: (174116,) | predictoras: ['M2', 'AMBIENTES', 'ANTIGUEDAD', 'BAÑOS', 'BAULERA', 'COCHERA', 'LAVADERO', 'TERRAZA', 'LAT', 'LON', 'BARRIO', 'ORIENT']
2026-11-09 22:54:25,885 | INFO     | src.features | Train: 139292 filas | Test: 34824 filas
2026-11-09 22:54:25,903 | INFO     | src.features | Target en train:
SEGMENTO
alto     0.333
medio    0.333
bajo     0.333
2026-11-09 22:54:25,915 | INFO     | src.features | Target en test:
SEGMENTO
alto     0.333
medio    0.333
bajo     0.333


(1000, 74) | NaN restantes: 0
['num__M2' 'num__AMBIENTES' 'num__ANTIGUEDAD' 'num__BAÑOS' 'num__BAULERA'
 'num__COCHERA' 'num__LAVADERO' 'num__TERRAZA' 'num__LAT' 'num__LON'
 'num__missingindicator_AMBIENTES' 'num__missingindicator_ANTIGUEDAD'
 'num__missingindicator_BAÑOS' 'num__missingindicator_BAULERA'
 'num__missingindicator_COCHERA']


In [ ]:
results = {}
for name in ["logreg", "random_forest"]:
    results[name] = tune_and_fit(name, X_train, y_train)
flush_mongo_logs()

2026-11-09 22:54:29,414 | INFO     | src.models | [logreg] búsqueda de hiperparámetros sobre 20000 filas (GridSearchCV, cv=3, scoring=f1_macro)
2026-11-09 22:54:49,092 | INFO     | src.models | [logreg] mejores parámetros: {'clf__C': 1} | f1_macro en CV: 0.6106 | 20 s
2026-11-09 22:55:08,841 | INFO     | src.models | [logreg] reentrenado con 139292 filas en 20 s
2026-11-09 22:55:09,266 | INFO     | src.models | [random_forest] búsqueda de hiperparámetros sobre 20000 filas (RandomizedSearchCV, cv=3, scoring=f1_macro)
2026-11-09 22:59:35,302 | INFO     | src.models | [random_forest] mejores parámetros: {'clf__n_estimators': 200, 'clf__min_samples_leaf': 10, 'clf__max_features': 0.5, 'clf__max_depth': None} | f1_macro en CV: 0.6365 | 266 s
